# MSENet — Six independent training-stability ablations (baseline, A1–A5)

**Dataset:** MiniImageNet (auto-downloaded from Google Drive) · **Evaluation:** 5-way 5-shot, Euclidean distance · **Protocol:** 10 epochs × 360 episodes (30-way 5-shot, 15 queries), unchanged from the baseline.

Every configuration is trained **independently from the same pretrained ResNet-18 weights, the same seed, the same pre-generated training episodes, and the same fixed validation / test episodes**. Each one changes **only** its own modification:

| Config | Change (and nothing else) |
|---|---|
| `baseline` | Original baseline code, unchanged |
| `A1` | BatchNorm2d modules forced to `eval()` after every `model.train()` (affine params stay trainable) |
| `A2` | Stem + layer1 frozen; layer3 lr 3e-6, layer4 lr 1e-5; attention heads + stage weights/gammas lr 5e-3 (layer2 lr 1e-6, see note) |
| `A3` | L2-SP: `(α/2)·‖θ_backbone − θ_pretrained‖²`, α = 1e-3 |
| `A4` | EMA (β = 0.99) of the complete model state; **EMA weights are used for validation / test** |
| `A5` | `CrossEntropyLoss(label_smoothing=0.1)` |

**Interpretation notes** (edit the constants in section 2 if you meant something else):
* A2: you did not specify layer2. It is set to **1e-6** (continuing the ×0.3 rule: 1e-5 → 3e-6 → ~1e-6). "Heads" = the randomly initialised self-attention modules, which get 5e-3 together with the special parameters (stage weights, γ1, γ2, unused cosine temp). Frozen BN layers inside stem/layer1 still update their running statistics (only A1 changes BN behaviour).
* A4: EMA is applied to every floating-point entry of the state dict (parameters **and** BN running statistics), updated after every optimizer step.
* Validation loss/accuracy and test accuracy are always measured on **5-way 5-shot** episodes; training accuracy is the 30-way episode accuracy of the model being trained.

**Engineering choices that do not change the maths:** images are decoded and resized (84×84) **once** and cached as uint8 in RAM (~1.3 GB); flip + normalisation are applied on the fly exactly as in the baseline. Checkpoints are kept in RAM only (no `.pth` files). Results are written to `/kaggle/working/msenet_results/` and zipped at the end (no automatic download).

In [ ]:
# ============================================================================
# 1. SETUP: packages, imports, reproducibility + memory helpers
# ============================================================================
import os, sys, gc, io, time, copy, json, math, random, shutil, hashlib, platform, tarfile, subprocess, warnings, contextlib, multiprocessing
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor


def _ensure(pip_name, import_name=None):
    try:
        __import__(import_name or pip_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

_ensure("easyfsl")
_ensure("gdown")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names
from easyfsl.samplers import TaskSampler
from tqdm.auto import tqdm
import matplotlib
import matplotlib.pyplot as plt
try:
    import psutil
except ImportError:
    psutil = None

# Suppress only known non-critical warnings (errors/exceptions are never hidden).
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=r".*lr_scheduler\.step\(\).*")
warnings.filterwarnings("ignore", message=r".*TypedStorage is deprecated.*")

# Same as baseline: 'spawn' start method (no worker processes are used, NUM_WORKERS = 0).
try:
    multiprocessing.set_start_method("spawn", force=True)
except RuntimeError:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("WARNING: no GPU detected. Select a GPU accelerator in Kaggle (Settings -> Accelerator); CPU would take days.")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()


def ram_report(tag):
    if psutil is None:
        return
    vm = psutil.virtual_memory()
    rss = psutil.Process().memory_info().rss / 2**30
    print(f"[RAM] {tag}: this process {rss:.2f} GB | system used {(vm.total - vm.available) / 2**30:.1f} / {vm.total / 2**30:.1f} GB")


def free_gpu():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


ENV_INFO = {
    "python": platform.python_version(), "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
}
try:
    import torchvision, easyfsl
    ENV_INFO["torchvision"] = torchvision.__version__
    ENV_INFO["easyfsl"] = getattr(easyfsl, "__version__", "unknown")
except Exception:
    pass
print("Device:", device, "| GPU:", ENV_INFO["gpu"])
print("Environment:", ENV_INFO)
ram_report("after imports")

In [ ]:
# ============================================================================
# 2. CONFIGURATION  (baseline constants copied verbatim from baseline.ipynb)
# ============================================================================
SEED = 42
# Fixed, pre-generated episodes shared by ALL configurations (fair comparison)
TRAIN_EPISODE_SEED, VAL_EPISODE_SEED, TEST_EPISODE_SEED = 20263, 20261, 20262
RESUME = True      # if the notebook is re-run in the same session, finished configs are reloaded (same settings only)

# ---------------- baseline experiment configuration ----------------
IMAGE_SIZE = 84
MAX_EPOCHS = 10
N_TRAINING_EPISODES = 360
N_VALIDATION_TASKS = 600
N_TEST_TASKS = 600
TRAIN_WAY = 30
TEST_WAY = 5
SHOT = 5
QUERY = 15
FEATURE_CHUNK_SIZE = 600
ATTENTION_CHUNK_SIZE = 600
DISTANCE_METRIC = 'euclidean'
COSINE_TEMP = 10.0
USE_AMP = True
STAGE_NODES   = ('maxpool', 'layer1', 'layer2', 'layer3', 'layer4')
STAGE_WEIGHTS = (1.0, 1.1, 1.2, 1.3, 1.4)
USE_IMAGENET_PRETRAINED = True
NUM_WORKERS = 0
gamma_init = 0.2
weights_lr = 0.005
backbone_lr = 1e-4
AMP_ON = bool(USE_AMP and device.type == "cuda")

# ---------------- ablation-specific settings ----------------
A2_LR_LAYER2 = 1e-6            # NOT specified in the request -> 1e-6 (continues the x0.3 rule). Edit if desired.
A2_LR_LAYER3 = 3e-6
A2_LR_LAYER4 = 1e-5
A2_LR_HEADS_AND_SPECIAL = 5e-3  # attention modules + stage weights + gammas (+ unused cosine temp)
A3_L2SP_ALPHA = 1e-3
A4_EMA_BETA = 0.99
A5_LABEL_SMOOTHING = 0.1

CONFIGS = {
    "baseline": dict(description="Original baseline (unchanged)", freeze_bn=False, discriminative_lr=False, l2sp_alpha=0.0, ema_beta=0.0, label_smoothing=0.0),
    "A1": dict(description="BatchNorm frozen (eval mode)", freeze_bn=True, discriminative_lr=False, l2sp_alpha=0.0, ema_beta=0.0, label_smoothing=0.0),
    "A2": dict(description="Discriminative LR + frozen stem/layer1", freeze_bn=False, discriminative_lr=True, l2sp_alpha=0.0, ema_beta=0.0, label_smoothing=0.0),
    "A3": dict(description=f"L2-SP regularisation (alpha={A3_L2SP_ALPHA:g})", freeze_bn=False, discriminative_lr=False, l2sp_alpha=A3_L2SP_ALPHA, ema_beta=0.0, label_smoothing=0.0),
    "A4": dict(description=f"EMA of weights (beta={A4_EMA_BETA}), EMA evaluated", freeze_bn=False, discriminative_lr=False, l2sp_alpha=0.0, ema_beta=A4_EMA_BETA, label_smoothing=0.0),
    "A5": dict(description=f"Label smoothing ({A5_LABEL_SMOOTHING})", freeze_bn=False, discriminative_lr=False, l2sp_alpha=0.0, ema_beta=0.0, label_smoothing=A5_LABEL_SMOOTHING),
}
CONFIG_ORDER = list(CONFIGS.keys())

SETTINGS_SIG = hashlib.sha1(json.dumps(dict(
    configs=CONFIGS, seed=SEED, eps=(TRAIN_EPISODE_SEED, VAL_EPISODE_SEED, TEST_EPISODE_SEED),
    base=(IMAGE_SIZE, MAX_EPOCHS, N_TRAINING_EPISODES, N_VALIDATION_TASKS, N_TEST_TASKS, TRAIN_WAY, TEST_WAY, SHOT, QUERY, DISTANCE_METRIC, USE_AMP, STAGE_WEIGHTS, gamma_init, weights_lr, backbone_lr),
    a2=(A2_LR_LAYER2, A2_LR_LAYER3, A2_LR_LAYER4, A2_LR_HEADS_AND_SPECIAL)), sort_keys=True, default=str).encode()).hexdigest()[:12]

# ---------------- paths ----------------
RUNNING_ON_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if RUNNING_ON_KAGGLE else Path("./working").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_BASE = WORK_DIR
if RUNNING_ON_KAGGLE:                       # /kaggle/temp is not saved as notebook output -> keeps the output small
    try:
        Path("/kaggle/temp").mkdir(parents=True, exist_ok=True)
        DATA_BASE = Path("/kaggle/temp")
    except OSError:
        pass
RESULTS_DIR = WORK_DIR / "msenet_results"
PLOTS_DIR = RESULTS_DIR / "plots"
STATE_DIR = WORK_DIR / "_msenet_state"      # tiny per-config json files (resume support), NOT included in the zip
for d in (RESULTS_DIR, PLOTS_DIR, STATE_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Settings signature: {SETTINGS_SIG} | AMP: {AMP_ON} | data dir: {DATA_BASE} | results dir: {RESULTS_DIR}")
print(pd.DataFrame(CONFIGS).T[["description"]].to_string())

In [ ]:
# ============================================================================
# 3. DATASET: automatic download (Google Drive) + extraction of train/val/test
# ============================================================================
import gdown

DRIVE_FOLDER = "https://drive.google.com/drive/folders/17a09kkqVivZQFggCw9I_YboJ23tcexNM"
DOWNLOAD_DIR = DATA_BASE / "miniimagenet_archives"
DATA_DIR = DATA_BASE / "miniimagenet"
DATA_DIR.mkdir(parents=True, exist_ok=True)
SPLITS = ("train", "val", "test")


def _marker(split):
    return DATA_DIR / f".{split}_extracted_ok"

def _find_archive(split):
    hits = list(DOWNLOAD_DIR.rglob(f"{split}.tar")) if DOWNLOAD_DIR.exists() else []
    return hits[0] if hits else None


missing = [s for s in SPLITS if not _marker(s).exists()]
if missing:
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, 4):
        if all(_find_archive(s) for s in missing):
            break
        print(f"Downloading MiniImageNet from Google Drive (attempt {attempt}/3) ...")
        try:
            gdown.download_folder(url=DRIVE_FOLDER, output=str(DOWNLOAD_DIR), quiet=False, use_cookies=False)
        except Exception as exc:                       # reported, then retried; final failure is raised below
            print("gdown error:", repr(exc))
    print("Downloaded files:")
    for p in sorted(DOWNLOAD_DIR.rglob("*")):
        if p.is_file():
            print(f"  {p.name}: {p.stat().st_size / 1e6:.1f} MB")

    for split in missing:
        archive = _find_archive(split)
        if archive is None:
            raise FileNotFoundError(f"{split}.tar was not found after 3 download attempts. Check that the Drive folder is publicly "
                                    "accessible, Kaggle internet is enabled, and the Drive download quota is not exhausted.")
        out_dir = DATA_DIR / split
        if out_dir.exists():
            shutil.rmtree(out_dir)                    # clean any partial earlier extraction
        out_dir.mkdir(parents=True)
        print(f"Extracting {archive.name} -> {out_dir} ...")
        with tarfile.open(archive, "r:*") as tar:
            try:
                tar.extractall(out_dir, filter="data")
            except TypeError:                         # very old Python without the `filter` argument
                tar.extractall(out_dir)
        _marker(split).touch()
        print(f"Finished extracting {split}")
    shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)   # free disk space
else:
    print("MiniImageNet already extracted - skipping download.")


def resolve_split_dir(root):
    """Descend through wrapper folders (e.g. train/train) until the class folders are reached."""
    p = Path(root)
    while True:
        entries = [e for e in p.iterdir() if not e.name.startswith(".")]
        subdirs = [e for e in entries if e.is_dir()]
        files = [e for e in entries if e.is_file()]
        if len(subdirs) == 1 and not files:
            p = subdirs[0]
            continue
        return p

TRAIN_DIR, VAL_DIR, TEST_DIR = (str(resolve_split_dir(DATA_DIR / s)) for s in SPLITS)
for split, path in zip(SPLITS, (TRAIN_DIR, VAL_DIR, TEST_DIR)):
    classes = [d for d in Path(path).iterdir() if d.is_dir()]
    print(f"{split}: {path} | class folders={len(classes)} | examples={[d.name for d in classes[:3]]}")

In [ ]:
# ============================================================================
# 4. RAM-CACHED DATASETS + FIXED EPISODES (shared by all six configurations)
# ============================================================================
_MEAN = torch.tensor((0.485, 0.456, 0.406)).view(3, 1, 1)     # identical to the baseline normalisation
_STD = torch.tensor((0.229, 0.224, 0.225)).view(3, 1, 1)
_resize = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))          # identical to the baseline Resize


class CachedImageDataset(Dataset):
    """Images are decoded + resized ONCE (uint8, ~1.3 GB for all splits). Per item: [random h-flip (train only)] -> /255 -> normalise,
    which is exactly the baseline pipeline Resize -> RandomHorizontalFlip -> ToTensor -> Normalize."""
    def __init__(self, images_u8, targets, classes, train):
        self.images, self.targets, self.classes, self.train = images_u8, [int(t) for t in targets], classes, train

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        x = self.images[idx]
        if self.train and torch.rand(1).item() < 0.5:
            x = x.flip(-1)
        x = (x.float().div_(255.0) - _MEAN) / _STD
        return x, self.targets[idx]

    def get_labels(self):                                       # used by the (patched) EasyFSL TaskSampler
        return self.targets


def build_cached_dataset(split_dir, train, desc):
    folder = ImageFolder(split_dir)                             # no transform: only paths / labels / PIL loader are used
    n = len(folder.samples)
    images = np.empty((n, 3, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)

    def _load(i):
        img = _resize(folder.loader(folder.samples[i][0]))
        images[i] = np.asarray(img, dtype=np.uint8).transpose(2, 0, 1)

    with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 4)) as ex:
        list(tqdm(ex.map(_load, range(n)), total=n, desc=desc, leave=False))
    return CachedImageDataset(torch.from_numpy(images), folder.targets, folder.classes, train)


train_set = build_cached_dataset(TRAIN_DIR, True, "Caching train")
val_set = build_cached_dataset(VAL_DIR, False, "Caching val")
test_set = build_cached_dataset(TEST_DIR, False, "Caching test")
print(f"Classes: train={len(train_set.classes)}, val={len(val_set.classes)}, test={len(test_set.classes)}")
print(f"Images: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}")
assert len(train_set.classes) >= TRAIN_WAY, "Not enough training classes for TRAIN_WAY."
assert len(val_set.classes) >= TEST_WAY, "Not enough validation classes for TEST_WAY."
assert len(test_set.classes) >= TEST_WAY, "Not enough test classes for TEST_WAY."
ram_report("after caching datasets")

# ---- Fast episode sampler compatibility patch for EasyFSL / newer PyTorch (verbatim from the baseline) ----
def _patched_task_sampler_init(self, dataset, n_way, n_shot, n_query, n_tasks):
    Sampler.__init__(self)
    self.n_way, self.n_shot, self.n_query, self.n_tasks = n_way, n_shot, n_query, n_tasks
    self.items_per_label = defaultdict(list)
    for item, label in enumerate(dataset.get_labels()):
        self.items_per_label[int(label)].append(item)
    if len(self.items_per_label) < n_way:
        raise ValueError(f'Requested {n_way}-way episodes but dataset has only {len(self.items_per_label)} classes.')
    required = n_shot + n_query
    too_small = [label for label, items in self.items_per_label.items() if len(items) < required]
    if too_small:
        raise ValueError(f'Classes have fewer than {required} images: {too_small[:5]}')

TaskSampler.__init__ = _patched_task_sampler_init
print('TaskSampler patched; labels are read from metadata only.')

# The (patched) baseline TaskSampler is kept ONLY for its `episodic_collate_fn`; the episodes themselves are pre-generated below.
collate_train = TaskSampler(train_set, n_way=TRAIN_WAY, n_shot=SHOT, n_query=QUERY, n_tasks=1)
collate_eval = TaskSampler(val_set, n_way=TEST_WAY, n_shot=SHOT, n_query=QUERY, n_tasks=1)


def generate_episodes(targets, n_way, n_shot, n_query, n_tasks, seed):
    """Same episode structure as EasyFSL's TaskSampler: n_way random classes, (n_shot + n_query) random images per class,
    class-major order (support first, then query, inside every class block). Deterministic given `seed`."""
    rng = np.random.default_rng(seed)
    items = defaultdict(list)
    for i, lab in enumerate(targets):
        items[int(lab)].append(i)
    labels = sorted(items)
    per_class = n_shot + n_query
    episodes = np.empty((n_tasks, n_way * per_class), dtype=np.int32)
    for t in range(n_tasks):
        chosen = rng.choice(len(labels), size=n_way, replace=False)
        episodes[t] = np.concatenate([rng.choice(items[labels[c]], size=per_class, replace=False) for c in chosen])
    return episodes


train_episodes = generate_episodes(train_set.targets, TRAIN_WAY, SHOT, QUERY, MAX_EPOCHS * N_TRAINING_EPISODES, TRAIN_EPISODE_SEED
                                   ).reshape(MAX_EPOCHS, N_TRAINING_EPISODES, -1)
val_episodes = generate_episodes(val_set.targets, TEST_WAY, SHOT, QUERY, N_VALIDATION_TASKS, VAL_EPISODE_SEED)
test_episodes = generate_episodes(test_set.targets, TEST_WAY, SHOT, QUERY, N_TEST_TASKS, TEST_EPISODE_SEED)


class FixedEpisodeSampler(Sampler):
    def __init__(self, episodes):
        self.episodes = episodes
    def __iter__(self):
        for row in self.episodes:
            yield row.tolist()
    def __len__(self):
        return len(self.episodes)


def make_loader(dataset, episodes, collate_sampler):
    loader = DataLoader(dataset, batch_sampler=FixedEpisodeSampler(episodes), num_workers=NUM_WORKERS,
                        pin_memory=(device.type == "cuda"), collate_fn=collate_sampler.episodic_collate_fn)
    assert len(loader) == len(episodes), f"Loader has {len(loader)} tasks, expected {len(episodes)}."
    return loader

def make_train_loader(epoch_index):
    return make_loader(train_set, train_episodes[epoch_index], collate_train)

val_loader = make_loader(val_set, val_episodes, collate_eval)
test_loader = make_loader(test_set, test_episodes, collate_eval)
print(f"Episodes per epoch: train={N_TRAINING_EPISODES}, val={len(val_loader)}, test={len(test_loader)}")


def check_episode(loader, way):
    s_img, s_lab, q_img, q_lab, _ = next(iter(loader))
    assert s_img.shape == (way * SHOT, 3, IMAGE_SIZE, IMAGE_SIZE), s_img.shape
    assert q_img.shape == (way * QUERY, 3, IMAGE_SIZE, IMAGE_SIZE), q_img.shape
    assert len(s_lab) == way * SHOT and len(q_lab) == way * QUERY
    assert len(set(s_lab.tolist())) == way
    assert (s_lab.view(way, SHOT) == s_lab.view(way, SHOT)[:, :1]).all(), "support labels must be class-major"
    print(f"{way}-way episode OK:", tuple(s_img.shape), tuple(q_img.shape))

t0 = time.time()
check_episode(make_train_loader(0), TRAIN_WAY)
check_episode(val_loader, TEST_WAY)
check_episode(test_loader, TEST_WAY)
print(f"Smoke test done in {time.time() - t0:.2f}s")

In [ ]:
# ============================================================================
# 5. BASELINE MODEL (verbatim from baseline.ipynb) + evaluation helpers
# ============================================================================
class Self_Attn(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        if in_dim < 8:
            raise ValueError(f'in_dim must be >= 8, got {in_dim}')
        self.query_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.key_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.value_conv = nn.Conv2d(in_dim, in_dim, kernel_size=1)

    def forward(self, x, gamma):
        b, c, h, w = x.shape
        n = h * w
        q = self.query_conv(x).reshape(b, -1, n).transpose(1, 2)  # B,N,C/8
        k = self.key_conv(x).reshape(b, -1, n)                   # B,C/8,N
        attention = torch.softmax(torch.bmm(q, k), dim=-1)        # B,N,N
        v = self.value_conv(x).reshape(b, c, n)                   # B,C,N
        out = torch.bmm(v, attention.transpose(1, 2)).reshape(b, c, h, w)
        return gamma * out + x

class PrototypicalNetworks(nn.Module):
    def __init__(self, backbone, image_size, stage_nodes=STAGE_NODES, weight=STAGE_WEIGHTS,
                 weightlearnable=True, gammalearnable=True, gamma1=0.2, gamma2=0.2,
                 feature_chunk_size=600, attention_chunk_size=600,
                 distance_metric='cosine', cosine_temp=10.0):
        super().__init__()
        if len(weight) != len(stage_nodes):
            raise ValueError(f'{len(weight)} weights given for {len(stage_nodes)} stages.')
        self.weightlearnable = weightlearnable
        self.gammalearnable = gammalearnable
        self.feature_chunk_size = feature_chunk_size
        self.attention_chunk_size = attention_chunk_size
        self.n_stages = len(stage_nodes)
        self.distance_metric = distance_metric
        
        # Learnable temperature scalar for cosine distance scaling
        self.cosine_temp = nn.Parameter(torch.tensor(float(cosine_temp)), requires_grad=True)

        self.stage_weights = nn.Parameter(torch.tensor(list(weight), dtype=torch.float32),
                                          requires_grad=weightlearnable)
        self.gamma1_l = nn.Parameter(torch.tensor(float(gamma1)), requires_grad=gammalearnable)
        self.gamma2_l = nn.Parameter(torch.tensor(float(gamma2)), requires_grad=gammalearnable)

        graph_nodes, _ = get_graph_node_names(backbone)
        return_nodes = {}
        for i, name in enumerate(stage_nodes):
            matches = [n for n in graph_nodes if n == name or n.startswith(name + '.')]
            if not matches:
                raise ValueError(f'Stage {name!r} not found in backbone graph.')
            return_nodes[matches[-1]] = f'stage_{i}'
        print('Stage nodes:', return_nodes)

        self.feature_extractor = create_feature_extractor(backbone, return_nodes=return_nodes)

        was_training = self.feature_extractor.training
        self.feature_extractor.eval()
        with torch.no_grad():
            dummy = self.feature_extractor(torch.zeros(1, 3, image_size, image_size))
        self.feature_extractor.train(was_training)
        channels = [dummy[f'stage_{i}'].shape[1] for i in range(self.n_stages)]
        print('Selected feature channels:', channels,
              '| spatial sizes:', [tuple(dummy[f'stage_{i}'].shape[2:]) for i in range(self.n_stages)])

        self.attention_modules = nn.ModuleList([Self_Attn(c) for c in channels])
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)

    def _extract_in_chunks(self, images):
        # Fast path: if batch fits in chunk_size, skip Python loop & dict rebuilding!
        if images.size(0) <= self.feature_chunk_size:
            return self.feature_extractor(images)
        collected = None
        for chunk in images.split(self.feature_chunk_size, dim=0):
            feats = self.feature_extractor(chunk)
            if collected is None:
                collected = {key: [value] for key, value in feats.items()}
            else:
                for key, value in feats.items(): collected[key].append(value)
        return {key: torch.cat(values, dim=0) for key, values in collected.items()}

    def _attend_and_pool(self, features, stage_idx, gamma):
        module = self.attention_modules[stage_idx]
        # Fast path: skip chunking loop when batch size <= attention_chunk_size
        if features.size(0) <= self.attention_chunk_size:
            return self.global_avg_pool(module(features, gamma)).flatten(1)
        pooled = []
        for chunk in features.split(self.attention_chunk_size, dim=0):
            pooled.append(self.global_avg_pool(module(chunk, gamma)).flatten(1))
        return torch.cat(pooled, dim=0)

    def _stage_distance(self, support_features, query_features, support_labels, stage_idx):
        z_support = self._attend_and_pool(support_features, stage_idx, self.gamma1_l)
        z_query = self._attend_and_pool(query_features, stage_idx, self.gamma2_l)
        
        # Vectorized prototype computation (faster than Python list comprehension over 30 labels)
        labels = torch.unique(support_labels, sorted=True)
        prototypes = torch.stack([z_support[support_labels == label].mean(0) for label in labels])

        if self.distance_metric == 'euclidean':
            return torch.cdist(z_query, prototypes, p=2)
            
        elif self.distance_metric == 'cosine':
            # L2-normalize query and prototype vectors onto unit hypersphere
            z_q_norm = nn.functional.normalize(z_query, p=2, dim=-1)
            proto_norm = nn.functional.normalize(prototypes, p=2, dim=-1)
            # Cosine distance = (1 - cosine_similarity) scaled by temperature
            cos_sim = torch.mm(z_q_norm, proto_norm.t())
            return self.cosine_temp * (1.0 - cos_sim)
            
        elif self.distance_metric == 'hybrid':
            # Combines L2-normalized Euclidean distance and Cosine distance
            euc_dist = torch.cdist(z_query, prototypes, p=2)
            z_q_norm = nn.functional.normalize(z_query, p=2, dim=-1)
            proto_norm = nn.functional.normalize(prototypes, p=2, dim=-1)
            cos_dist = self.cosine_temp * (1.0 - torch.mm(z_q_norm, proto_norm.t()))
            return 0.5 * euc_dist + 0.5 * cos_dist
        else:
            raise ValueError(f"Unknown distance_metric: {self.distance_metric}")

    def forward(self, support_images, support_labels, query_images):
        support_feats = self._extract_in_chunks(support_images)
        query_feats = self._extract_in_chunks(query_images)
        distances = torch.stack([
            self._stage_distance(support_feats[f'stage_{i}'], query_feats[f'stage_{i}'], support_labels, i)
            for i in range(self.n_stages)
        ])
        return -(self.stage_weights.view(-1, 1, 1) * distances).sum(dim=0)

print('Model class defined.')


# ---------------- evaluation (baseline logic + loss, per-episode accuracy and optional forward-pass timing) ----------------
@torch.no_grad()
def evaluate(data_loader, model, desc='Evaluating', timed=False):
    model.eval()
    correct_total, prediction_total, n_tasks, loss_sum = 0, 0, 0, 0.0
    ep_acc, ep_time = [], []
    if timed:   # warm-up (cudnn / allocator) on the first episode; not counted
        s0, l0, q0, _, _ = next(iter(data_loader))
        for _ in range(3):
            model(s0.to(device), l0.to(device), q0.to(device))
        sync()
    for support_images, support_labels, query_images, query_labels, _ in tqdm(data_loader, total=len(data_loader), desc=desc, leave=False):
        s_img, s_lab, q_img = support_images.to(device, non_blocking=True), support_labels.to(device), query_images.to(device, non_blocking=True)
        labels = query_labels.to(device)
        if timed:
            sync(); t0 = time.perf_counter()
        scores = model(s_img, s_lab, q_img)
        if timed:
            sync(); ep_time.append(time.perf_counter() - t0)
        correct = (scores.argmax(dim=1) == labels).sum().item()
        correct_total += correct
        prediction_total += labels.numel()
        ep_acc.append(100.0 * correct / labels.numel())
        loss_sum += F.cross_entropy(scores.float(), labels).item()
        n_tasks += 1
    if prediction_total == 0:
        raise RuntimeError('Evaluation loader produced no query examples.')
    if n_tasks != len(data_loader):
        raise RuntimeError(f'Evaluated {n_tasks} tasks but loader promised {len(data_loader)}.')
    return dict(acc=100.0 * correct_total / prediction_total, loss=loss_sum / n_tasks,
                per_episode_acc=ep_acc, per_episode_time=ep_time)

In [ ]:
# ============================================================================
# 6. EXPERIMENT MACHINERY: pretrained cache, model/optimizer builders, EMA, one-config runner
# ============================================================================
def _load_init_backbone_state():
    """Pretrained ResNet-18 weights are downloaded ONCE and cached in RAM; every config starts from this exact state."""
    sd, pretrained = None, False
    if USE_IMAGENET_PRETRAINED:
        try:
            sd = {k: v.detach().cpu().clone() for k, v in resnet18(weights=ResNet18_Weights.DEFAULT).state_dict().items()}
            pretrained = True
        except Exception as exc:
            print('Could not load torchvision pretrained weights; using random initialization:', exc)
    if sd is None:
        set_seed(SEED)
        sd = {k: v.detach().cpu().clone() for k, v in resnet18(weights=None).state_dict().items()}
    return sd, pretrained

INIT_BACKBONE_SD, USING_PRETRAINED = _load_init_backbone_state()
print("Pretrained ImageNet ResNet-18 cached in RAM:", USING_PRETRAINED)
ENV_INFO["pretrained_backbone"] = USING_PRETRAINED


def build_model(verbose=False):
    set_seed(SEED)                                   # identical init of the randomly initialised attention modules in every config
    backbone = resnet18(weights=None)
    backbone.load_state_dict(INIT_BACKBONE_SD)
    backbone.fc = nn.Identity()
    sink = contextlib.nullcontext() if verbose else contextlib.redirect_stdout(io.StringIO())
    with sink:
        model = PrototypicalNetworks(
            backbone=backbone, image_size=IMAGE_SIZE, stage_nodes=STAGE_NODES, weight=STAGE_WEIGHTS,
            weightlearnable=True, gammalearnable=True, gamma1=gamma_init, gamma2=gamma_init,
            feature_chunk_size=FEATURE_CHUNK_SIZE, attention_chunk_size=ATTENTION_CHUNK_SIZE,
            distance_metric=DISTANCE_METRIC, cosine_temp=COSINE_TEMP,
        )
    return model.to(device)


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def build_optimizer(model, cfg):
    special = [model.stage_weights, model.gamma1_l, model.gamma2_l, model.cosine_temp]
    special_ids = {id(p) for p in special}
    named = [(n, p) for n, p in model.named_parameters() if id(p) not in special_ids]
    info = []
    if not cfg["discriminative_lr"]:
        # ---- exactly the baseline parameter groups ----
        regular = [p for _, p in named]
        groups = [{'params': regular, 'lr': backbone_lr}, {'params': [model.cosine_temp], 'lr': weights_lr}]
        info.append(f"regular (backbone + attention): {sum(p.numel() for p in regular)/1e6:.3f}M params @ lr {backbone_lr:g}")
        if model.weightlearnable:
            groups.append({'params': [model.stage_weights], 'lr': weights_lr})
        if model.gammalearnable:
            groups.append({'params': [model.gamma1_l, model.gamma2_l], 'lr': weights_lr})
        info.append(f"cosine_temp / stage_weights / gammas @ lr {weights_lr:g}")
    else:
        # ---- A2: discriminative learning rates ----
        buckets = {"frozen": [], "layer2": [], "layer3": [], "layer4": [], "heads": []}
        for n, p in named:
            if n.startswith("feature_extractor."):
                top = n.split(".")[1]
                if top in ("conv1", "bn1", "layer1"):
                    p.requires_grad_(False)
                    buckets["frozen"].append(p)
                elif top in ("layer2", "layer3", "layer4"):
                    buckets[top].append(p)
                else:
                    raise RuntimeError(f"Unexpected backbone parameter: {n}")
            elif n.startswith("attention_modules."):
                buckets["heads"].append(p)
            else:
                raise RuntimeError(f"Unassigned parameter: {n}")
        for k, v in buckets.items():
            if not v:
                raise RuntimeError(f"A2 bucket '{k}' is empty - parameter names did not match the expected ResNet layout.")
        lr_map = {"layer2": A2_LR_LAYER2, "layer3": A2_LR_LAYER3, "layer4": A2_LR_LAYER4, "heads": A2_LR_HEADS_AND_SPECIAL}
        groups = [{'params': buckets[k], 'lr': lr_map[k]} for k in ("layer2", "layer3", "layer4", "heads")]
        groups.append({'params': [model.cosine_temp], 'lr': A2_LR_HEADS_AND_SPECIAL})
        if model.weightlearnable:
            groups.append({'params': [model.stage_weights], 'lr': A2_LR_HEADS_AND_SPECIAL})
        if model.gammalearnable:
            groups.append({'params': [model.gamma1_l, model.gamma2_l], 'lr': A2_LR_HEADS_AND_SPECIAL})
        info.append(f"FROZEN stem+layer1: {sum(p.numel() for p in buckets['frozen'])/1e6:.3f}M params")
        for k in ("layer2", "layer3", "layer4", "heads"):
            info.append(f"{k}: {sum(p.numel() for p in buckets[k])/1e6:.3f}M params @ lr {lr_map[k]:g}")
        info.append(f"cosine_temp / stage_weights / gammas @ lr {A2_LR_HEADS_AND_SPECIAL:g}")
        covered = {id(p) for g in groups for p in g['params']}
        orphans = [n for n, p in model.named_parameters() if p.requires_grad and id(p) not in covered]
        if orphans:
            raise RuntimeError(f"Trainable parameters missing from the optimizer: {orphans[:5]}")
    return optim.Adam(groups), info


def set_train_mode(model, freeze_bn):
    model.train()
    if freeze_bn:                                    # A1: BatchNorm2d stays in eval mode (running stats used, not updated); affine params still train
        for m in model.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()


class ModelEMA:
    """EMA of the complete model state (every floating-point entry of the state dict: parameters AND BN running statistics)."""
    def __init__(self, model, beta):
        self.beta = beta
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        ema_sd, live_sd = self.module.state_dict(), model.state_dict()      # tensors share storage with the real parameters / buffers
        fkeys = [k for k, v in ema_sd.items() if v.is_floating_point()]
        okeys = [k for k in ema_sd if k not in set(fkeys)]
        self.ema_f, self.live_f = [ema_sd[k] for k in fkeys], [live_sd[k] for k in fkeys]
        self.ema_o, self.live_o = [ema_sd[k] for k in okeys], [live_sd[k] for k in okeys]

    @torch.no_grad()
    def update(self, model=None):
        torch._foreach_mul_(self.ema_f, self.beta)
        torch._foreach_add_(self.ema_f, self.live_f, alpha=1.0 - self.beta)
        for e, l in zip(self.ema_o, self.live_o):
            e.copy_(l)


def count_gflops(model):
    try:
        from torch.utils.flop_counter import FlopCounterMode
        model.eval()
        s = torch.randn(TEST_WAY * SHOT, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
        sl = torch.arange(TEST_WAY, device=device).repeat_interleave(SHOT)
        q = torch.randn(TEST_WAY * QUERY, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
        with torch.no_grad(), FlopCounterMode(display=False) as fc:
            model(s, sl, q)
        return fc.get_total_flops() / 1e9
    except Exception as exc:
        print("FLOP counting unavailable:", repr(exc))
        return float("nan")


def learned_param_stats(model):
    """Compact statistics of what was learned (computed on the model that is actually tested)."""
    st = {}
    sw = model.stage_weights.detach().cpu().tolist()
    for i, v in enumerate(sw, 1):
        st[f"stage_weight_{i}"] = v
        st[f"stage_weight_share_{i}"] = v / sum(sw)
    st["gamma1"], st["gamma2"] = model.gamma1_l.item(), model.gamma2_l.item()
    st["cosine_temp_unused_with_euclidean"] = model.cosine_temp.item()
    diff2, ref2 = defaultdict(float), defaultdict(float)
    for n, p in model.named_parameters():
        if n.startswith("feature_extractor."):
            key = n[len("feature_extractor."):]
            ref = INIT_BACKBONE_SD[key].float()
            top = key.split(".")[0]
            grp = "stem" if top in ("conv1", "bn1") else top
            diff2[grp] += ((p.detach().float().cpu() - ref) ** 2).sum().item()
            ref2[grp] += (ref ** 2).sum().item()
    for grp in ("stem", "layer1", "layer2", "layer3", "layer4"):
        st[f"backbone_rel_drift_{grp}"] = math.sqrt(diff2[grp] / max(ref2[grp], 1e-12))
    st["backbone_rel_drift_all"] = math.sqrt(sum(diff2.values()) / max(sum(ref2.values()), 1e-12))
    for i, m in enumerate(model.attention_modules, 1):
        st[f"attn{i}_query_w_norm"] = m.query_conv.weight.detach().norm().item()
        st[f"attn{i}_key_w_norm"] = m.key_conv.weight.detach().norm().item()
        st[f"attn{i}_value_w_norm"] = m.value_conv.weight.detach().norm().item()
    return st


def _mb(x):
    return x / 2**20


def run_experiment(name, cfg):
    print("=" * 100)
    print(f"CONFIG {name}: {cfg['description']}")
    print("=" * 100)
    free_gpu()
    wall0 = time.perf_counter()

    # ---------------- build ----------------
    model = build_model(verbose=(name == "baseline"))
    optimizer, group_info = build_optimizer(model, cfg)
    params_total, params_trainable = count_params(model)
    for line in group_info:
        print("   ", line)
    print(f"    parameters: total={params_total/1e6:.3f}M | trainable={params_trainable/1e6:.3f}M")

    sp_params = sp_ref = None
    extra_state_params = 0
    if cfg["l2sp_alpha"] > 0:                         # A3: reference = original pretrained backbone weights
        sp_params = [p for n, p in model.named_parameters() if n.startswith("feature_extractor.")]
        sp_ref = [p.detach().clone() for p in sp_params]
        extra_state_params += sum(p.numel() for p in sp_ref)
    ema = None
    if cfg["ema_beta"] > 0:                           # A4
        ema = ModelEMA(model, cfg["ema_beta"])
        extra_state_params += sum(p.numel() for p in ema.module.parameters())
    eval_model = ema.module if ema is not None else model

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ON)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])

    # ---------------- train ----------------
    set_seed(SEED + 1)                                # identical horizontal-flip stream in every config
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    best_val, best_epoch, best_state = -1.0, 0, None
    history, train_time_total, val_time_total = [], 0.0, 0.0

    for epoch in range(1, MAX_EPOCHS + 1):
        set_train_mode(model, cfg["freeze_bn"])
        loader = make_train_loader(epoch - 1)
        stats = torch.zeros(4, device=device)         # [plain CE, objective CE, regulariser, accuracy] summed over episodes
        n_episodes = 0
        progress = tqdm(loader, total=len(loader), desc=f"[{name}] epoch {epoch}/{MAX_EPOCHS}", leave=False)
        sync(); t0 = time.perf_counter()
        for it, (support_images, support_labels, query_images, query_labels, _) in enumerate(progress):
            optimizer.zero_grad(set_to_none=True)
            q_lab = query_labels.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=AMP_ON):
                scores = model(support_images.to(device, non_blocking=True), support_labels.to(device),
                               query_images.to(device, non_blocking=True))
                loss_ce = criterion(scores, q_lab)
            loss = loss_ce
            reg = torch.zeros((), device=device)
            if sp_params is not None:                 # A3: (alpha/2) * || theta - theta_pretrained ||^2  (fp32, outside autocast)
                reg = 0.5 * cfg["l2sp_alpha"] * sum(((p - r) ** 2).sum() for p, r in zip(sp_params, sp_ref))
                loss = loss_ce + reg
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            if ema is not None:
                ema.update()
            with torch.no_grad():
                sc = scores.detach().float()
                stats += torch.stack([F.cross_entropy(sc, q_lab), loss_ce.detach().float(), reg.detach().float(),
                                      (sc.argmax(dim=1) == q_lab).float().mean()])
            n_episodes += 1
            if it % 30 == 0:
                progress.set_postfix(loss=f"{loss_ce.item():.4f}")
        sync(); epoch_train_time = time.perf_counter() - t0
        if n_episodes != N_TRAINING_EPISODES:
            raise RuntimeError(f'Ran {n_episodes} episodes, config says {N_TRAINING_EPISODES}.')
        lr_now = optimizer.param_groups[0]["lr"]
        scheduler.step()
        train_time_total += epoch_train_time
        ce_plain, ce_obj, reg_v, tr_acc = (stats / n_episodes).tolist()

        # ---------------- validation (EMA weights for A4) ----------------
        sync(); tv0 = time.perf_counter()
        val = evaluate(val_loader, eval_model, desc=f"[{name}] validating epoch {epoch}")
        sync(); epoch_val_time = time.perf_counter() - tv0
        val_time_total += epoch_val_time
        val_raw = evaluate(val_loader, model, desc=f"[{name}] validating raw model")["acc"] if ema is not None else float("nan")

        row = dict(config=name, epoch=epoch, train_acc=100.0 * tr_acc, train_loss=ce_plain, train_loss_objective=ce_obj + reg_v,
                   train_reg=reg_v, val_acc=val["acc"], val_loss=val["loss"], val_acc_raw_model=val_raw, lr_group0=lr_now,
                   epoch_train_time_s=epoch_train_time, epoch_val_time_s=epoch_val_time,
                   gpu_peak_alloc_mb=_mb(torch.cuda.max_memory_allocated()) if device.type == "cuda" else 0.0,
                   gamma1=model.gamma1_l.item(), gamma2=model.gamma2_l.item(), cosine_temp_unused=model.cosine_temp.item())
        for i, v in enumerate(model.stage_weights.detach().cpu().tolist(), 1):
            row[f"stage_weight_{i}"] = v
        history.append(row)
        extra = f", val(raw model)={val_raw:.2f}%" if ema is not None else ""
        print(f"[{name}] epoch {epoch}: train CE={ce_plain:.4f}, train acc(30-way)={100*tr_acc:.2f}%, val acc(5-way)={val['acc']:.2f}%{extra}, "
              f"val loss={val['loss']:.4f}, train time={epoch_train_time/60:.2f} min")
        if val["acc"] > best_val:
            best_val, best_epoch = val["acc"], epoch
            best_state = {k: v.detach().cpu().clone() for k, v in eval_model.state_dict().items()}   # RAM only, no .pth on disk

    peak_train_alloc = _mb(torch.cuda.max_memory_allocated()) if device.type == "cuda" else 0.0
    peak_train_reserved = _mb(torch.cuda.max_memory_reserved()) if device.type == "cuda" else 0.0
    print(f"[{name}] best validation accuracy {best_val:.2f}% at epoch {best_epoch}")

    # ---------------- release ALL training objects, then test a clean copy of the best-val model ----------------
    del scores, sc, loss, loss_ce, reg, q_lab, progress                      # these still reference the autograd graph / GPU activations
    del model, optimizer, scheduler, scaler, sp_params, sp_ref, ema, eval_model, trainable_params, criterion, loader, stats
    free_gpu()
    infer_model = build_model()
    infer_model.load_state_dict(best_state)
    del best_state
    free_gpu()                                        # resets peak statistics -> inference memory excludes training state
    test = evaluate(test_loader, infer_model, desc=f"[{name}] testing", timed=True)
    peak_infer_alloc = _mb(torch.cuda.max_memory_allocated()) if device.type == "cuda" else 0.0
    peak_infer_reserved = _mb(torch.cuda.max_memory_reserved()) if device.type == "cuda" else 0.0
    gflops = count_gflops(infer_model)
    stats_final = learned_param_stats(infer_model)
    params_total_infer, _ = count_params(infer_model)
    del infer_model
    free_gpu()

    t = np.array(test["per_episode_time"]) * 1000.0
    n_query = TEST_WAY * QUERY
    summary = dict(
        config=name, description=cfg["description"], best_epoch=best_epoch, best_val_acc=best_val,
        test_acc=test["acc"], test_loss=test["loss"],
        final_train_acc=history[-1]["train_acc"], final_val_acc=history[-1]["val_acc"],
        final_train_loss=history[-1]["train_loss"], final_val_loss=history[-1]["val_loss"],
        train_time_s=train_time_total, train_time_per_epoch_s=train_time_total / MAX_EPOCHS,
        train_ms_per_episode=1000.0 * train_time_total / (MAX_EPOCHS * N_TRAINING_EPISODES),
        val_time_total_s=val_time_total, total_wall_time_s=time.perf_counter() - wall0,
        infer_ms_per_episode_mean=float(t.mean()), infer_ms_per_episode_median=float(np.median(t)), infer_ms_per_episode_std=float(t.std(ddof=1)),
        infer_ms_per_query_image=float(t.mean() / n_query), infer_test_total_forward_s=float(t.sum() / 1000.0),
        params_total=params_total_infer, params_trainable=params_trainable, params_total_M=params_total_infer / 1e6, params_trainable_M=params_trainable / 1e6,
        extra_train_state_params_M=extra_state_params / 1e6,
        gflops_per_5way5shot_episode=gflops,
        peak_gpu_train_alloc_mb=peak_train_alloc, peak_gpu_train_reserved_mb=peak_train_reserved,
        peak_gpu_infer_alloc_mb=peak_infer_alloc, peak_gpu_infer_reserved_mb=peak_infer_reserved,
        process_rss_gb=(psutil.Process().memory_info().rss / 2**30) if psutil else float("nan"),
    )
    print(f"[{name}] TEST ACCURACY (5-way {SHOT}-shot, {N_TEST_TASKS} fixed episodes): {test['acc']:.2f}% | "
          f"train time {train_time_total/60:.1f} min | inference {summary['infer_ms_per_episode_mean']:.1f} ms/episode")
    return dict(name=name, cfg=cfg, history=history, summary=summary, per_episode_acc=list(test["per_episode_acc"]), stats=stats_final)

In [ ]:
# ============================================================================
# 7. RUN ALL SIX CONFIGURATIONS (each one independent: fresh model, optimizer, RNG state; memory released in between)
# ============================================================================
RESULTS = {}
t_all = time.perf_counter()
for name, cfg in CONFIGS.items():
    state_file = STATE_DIR / f"{name}.json"
    if RESUME and state_file.exists():
        saved = json.loads(state_file.read_text())
        if saved.get("signature") == SETTINGS_SIG:
            print(f"[{name}] finished earlier in this session with identical settings -> loaded from {state_file.name}")
            RESULTS[name] = saved["result"]
            continue
    result = run_experiment(name, cfg)
    state_file.write_text(json.dumps(dict(signature=SETTINGS_SIG, result=result)))
    RESULTS[name] = result
    free_gpu()
    ram_report(f"after {name}")

print(f"\nAll configurations finished in {(time.perf_counter() - t_all) / 60:.1f} min")
print(pd.DataFrame([{"config": n, "best_val": r["summary"]["best_val_acc"], "test_acc": r["summary"]["test_acc"]} for n, r in RESULTS.items()]).round(2).to_string(index=False))

In [ ]:
# ============================================================================
# 8. CSV REPORTS
# ============================================================================
for f in list(RESULTS_DIR.rglob("*")):
    if f.is_file() and f.suffix in (".csv", ".png"):
        f.unlink()                                    # remove stale files from earlier runs

order = [n for n in CONFIG_ORDER if n in RESULTS]
base_ep = np.array(RESULTS["baseline"]["per_episode_acc"])
n_ep = len(base_ep)

main_rows, eff_rows = [], []
for n in order:
    r = RESULTS[n]; s = r["summary"]
    ep = np.array(r["per_episode_acc"])
    ci = 1.96 * ep.std(ddof=1) / math.sqrt(len(ep))
    d = ep - base_ep
    d_ci = 1.96 * d.std(ddof=1) / math.sqrt(len(d)) if n != "baseline" else 0.0
    main_rows.append({
        "config": n, "description": s["description"],
        "test_acc_5way5shot": ep.mean(), "test_acc_ci95": ci, "test_episode_std": ep.std(ddof=1),
        "delta_vs_baseline": d.mean(), "delta_ci95_paired": d_ci,
        "delta_significant_95": bool(n != "baseline" and abs(d.mean()) > d_ci),
        "best_val_acc": s["best_val_acc"], "best_epoch": s["best_epoch"],
        "final_epoch_train_acc_30way": s["final_train_acc"], "final_epoch_val_acc_5way": s["final_val_acc"],
        "final_train_ce": s["final_train_loss"], "final_val_loss": s["final_val_loss"],
        "train_time_min": s["train_time_s"] / 60, "infer_ms_per_episode": s["infer_ms_per_episode_mean"],
        "params_M": s["params_total_M"], "trainable_params_M": s["params_trainable_M"],
        "peak_gpu_train_mb": s["peak_gpu_train_alloc_mb"], "peak_gpu_infer_mb": s["peak_gpu_infer_alloc_mb"],
        "gflops_per_episode": s["gflops_per_5way5shot_episode"],
    })
    eff_rows.append({k: v for k, v in s.items() if k not in ("description",)})

main_df = pd.DataFrame(main_rows)
eff_df = pd.DataFrame(eff_rows)
epoch_df = pd.concat([pd.DataFrame(RESULTS[n]["history"]) for n in order], ignore_index=True)
sw_df = pd.DataFrame([{"config": n, **{f"stage_weight_{i}": RESULTS[n]["stats"][f"stage_weight_{i}"] for i in range(1, 6)},
                       **{f"stage_weight_share_{i}": RESULTS[n]["stats"][f"stage_weight_share_{i}"] for i in range(1, 6)},
                       "source": "best-validation checkpoint (the model that was tested)"} for n in order])
gm_df = pd.DataFrame([{"config": n, "gamma1_support": RESULTS[n]["stats"]["gamma1"], "gamma2_query": RESULTS[n]["stats"]["gamma2"],
                       "cosine_temp_unused_with_euclidean": RESULTS[n]["stats"]["cosine_temp_unused_with_euclidean"],
                       "note": "no learnable softmax temperature exists in these six configs; cosine_temp is inert for Euclidean distance"} for n in order])
stats_df = pd.DataFrame([{"config": n, **RESULTS[n]["stats"]} for n in order])
per_ep_df = pd.DataFrame({n: RESULTS[n]["per_episode_acc"] for n in order}); per_ep_df.insert(0, "test_episode", range(1, n_ep + 1))
env_rows = [{"key": k, "value": v} for k, v in ENV_INFO.items()] + [
    {"key": "settings_signature", "value": SETTINGS_SIG}, {"key": "seed", "value": SEED},
    {"key": "episode_seeds(train,val,test)", "value": (TRAIN_EPISODE_SEED, VAL_EPISODE_SEED, TEST_EPISODE_SEED)},
    {"key": "epochs x episodes", "value": f"{MAX_EPOCHS} x {N_TRAINING_EPISODES}"}, {"key": "train_way/test_way/shot/query", "value": (TRAIN_WAY, TEST_WAY, SHOT, QUERY)},
    {"key": "val_tasks/test_tasks", "value": (N_VALIDATION_TASKS, N_TEST_TASKS)}, {"key": "distance", "value": DISTANCE_METRIC}, {"key": "amp", "value": AMP_ON},
    {"key": "A2 lrs (layer2,layer3,layer4,heads+special)", "value": (A2_LR_LAYER2, A2_LR_LAYER3, A2_LR_LAYER4, A2_LR_HEADS_AND_SPECIAL)},
    {"key": "A3 alpha / A4 beta / A5 smoothing", "value": (A3_L2SP_ALPHA, A4_EMA_BETA, A5_LABEL_SMOOTHING)},
    {"key": "train_time_note", "value": "training loop only (data fetch + forward/backward + optimizer + EMA/L2-SP), validation excluded"},
    {"key": "inference_time_note", "value": "model forward only per 5-way 5-shot episode (25 support + 75 query), CUDA-synchronised, 3 warm-up passes excluded"},
    {"key": "seeds_note", "value": "one training seed per config: CIs reflect episode sampling only, not training-seed variance"},
]
env_df = pd.DataFrame(env_rows)

reports = {"main_comparison.csv": main_df, "per_epoch_metrics.csv": epoch_df, "efficiency_metrics.csv": eff_df,
           "final_stage_weights.csv": sw_df, "final_gamma_temperature.csv": gm_df, "learned_param_stats.csv": stats_df,
           "per_episode_test_accuracy.csv": per_ep_df, "run_environment.csv": env_df}
for fname, df in reports.items():
    df.to_csv(RESULTS_DIR / fname, index=False, float_format="%.6g")
    print(f"saved {fname}  ({len(df)} rows)")

show_cols = ["config", "test_acc_5way5shot", "test_acc_ci95", "delta_vs_baseline", "delta_ci95_paired", "delta_significant_95",
             "best_val_acc", "best_epoch", "train_time_min", "infer_ms_per_episode", "params_M", "trainable_params_M", "peak_gpu_train_mb"]
print("\n" + main_df[show_cols].round(3).to_string(index=False))

In [ ]:
# ============================================================================
# 9. PLOTS (all saved as PNG in msenet_results/plots)
# ============================================================================
PALETTE = ["#4c78a8", "#f58518", "#54a24b", "#e45756", "#72b7b2", "#b279a2"]
COLORS = {n: PALETTE[i % len(PALETTE)] for i, n in enumerate(CONFIG_ORDER)}
LABELS = {n: (n if n == "baseline" else f"{n}") for n in CONFIG_ORDER}
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

def _save(fig, fname):
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / fname, dpi=130, bbox_inches="tight")
    plt.show(); plt.close(fig)
    print("saved", fname)

def line_plot(col, title, ylabel, fname, extra_raw=False):
    fig, ax = plt.subplots(figsize=(8, 5))
    for n in order:
        d = epoch_df[epoch_df.config == n]
        ax.plot(d.epoch, d[col], marker="o", ms=4, lw=1.8, label=LABELS[n], color=COLORS[n])
    if extra_raw and "A4" in order:
        d = epoch_df[epoch_df.config == "A4"]
        ax.plot(d.epoch, d.val_acc_raw_model, ls="--", lw=1.2, color=COLORS["A4"], alpha=0.7, label="A4 (raw, non-EMA model)")
    ax.set_title(title); ax.set_xlabel("epoch"); ax.set_ylabel(ylabel); ax.set_xticks(range(1, MAX_EPOCHS + 1)); ax.legend(ncol=2, fontsize=9)
    _save(fig, fname)

def bar_plot(values, errs, title, ylabel, fname, fmt="%.2f", zoom=False):
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(order, values, yerr=errs, capsize=4, color=[COLORS[n] for n in order])
    ax.bar_label(bars, fmt=fmt, padding=3, fontsize=9)
    if zoom:
        lo, hi = min(values), max(values)
        ax.set_ylim(lo - max(1.0, 3 * (hi - lo)), hi + max(0.8, 1.5 * (hi - lo)))
    ax.set_title(title); ax.set_ylabel(ylabel); ax.grid(axis="x", visible=False)
    _save(fig, fname)

line_plot("train_acc", "Training accuracy vs epoch (30-way 5-shot episodes, model being trained)", "accuracy (%)", "01_train_accuracy_vs_epoch.png")
line_plot("train_loss", "Training loss vs epoch (plain cross-entropy; no smoothing / regulariser)", "cross-entropy", "02_train_loss_vs_epoch.png")
line_plot("val_acc", "Validation accuracy vs epoch (5-way 5-shot, fixed 600 episodes; A4 = EMA weights)", "accuracy (%)", "03_val_accuracy_vs_epoch.png", extra_raw=True)
line_plot("val_loss", "Validation loss vs epoch (5-way, plain cross-entropy)", "cross-entropy", "04_val_loss_vs_epoch.png")

bar_plot(main_df.test_acc_5way5shot.values, main_df.test_acc_ci95.values, "Test accuracy, 5-way 5-shot (best-validation checkpoint, 95% CI over 600 episodes)", "accuracy (%)", "05_test_accuracy_comparison.png", zoom=True)
bar_plot(eff_df.train_time_s.values / 60, None, "Training time (10 epochs, validation excluded)", "minutes", "06_training_time_comparison.png", fmt="%.1f")
bar_plot(eff_df.infer_ms_per_episode_mean.values, eff_df.infer_ms_per_episode_std.values, "Inference time per 5-way 5-shot episode (forward pass only, mean ± std)", "ms / episode", "07_inference_time_comparison.png", fmt="%.1f")

fig, ax = plt.subplots(figsize=(9, 5)); x = np.arange(len(order)); w = 0.38
b1 = ax.bar(x - w / 2, eff_df.peak_gpu_train_alloc_mb, w, label="training (peak allocated)", color="#4c78a8")
b2 = ax.bar(x + w / 2, eff_df.peak_gpu_infer_alloc_mb, w, label="inference (peak allocated)", color="#f58518")
ax.bar_label(b1, fmt="%.0f", fontsize=8, padding=2); ax.bar_label(b2, fmt="%.0f", fontsize=8, padding=2)
ax.set_xticks(x); ax.set_xticklabels(order); ax.set_ylabel("MB"); ax.set_title("Peak GPU memory"); ax.legend(); ax.grid(axis="x", visible=False)
_save(fig, "08_gpu_memory_comparison.png")

# 09a: stage-weight trajectories (epoch 0 = initial values)
ncol = 3; nrow = int(math.ceil(len(order) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3.6 * nrow), sharey=True); axes = np.atleast_1d(axes).ravel()
stage_colors = plt.cm.viridis(np.linspace(0, 0.9, 5))
for ax, n in zip(axes, order):
    d = epoch_df[epoch_df.config == n]
    for i in range(5):
        ax.plot([0] + d.epoch.tolist(), [STAGE_WEIGHTS[i]] + d[f"stage_weight_{i + 1}"].tolist(), marker="o", ms=3, color=stage_colors[i], label=f"w{i + 1} ({STAGE_NODES[i]})")
    ax.set_title(n); ax.set_xlabel("epoch")
axes[0].set_ylabel("stage weight"); axes[0].legend(fontsize=7)
for ax in axes[len(order):]:
    ax.axis("off")
fig.suptitle("Learned stage-weight trajectories (live model, end of each epoch)")
_save(fig, "09a_stage_weight_trajectories.png")

# 09b: gamma trajectories
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, col, ttl in zip(axes, ["gamma1", "gamma2"], ["gamma1 (support attention)", "gamma2 (query attention)"]):
    for n in order:
        d = epoch_df[epoch_df.config == n]
        ax.plot([0] + d.epoch.tolist(), [gamma_init] + d[col].tolist(), marker="o", ms=3, label=n, color=COLORS[n])
    ax.set_title(ttl); ax.set_xlabel("epoch")
axes[0].set_ylabel("value"); axes[0].legend(fontsize=8, ncol=2)
fig.suptitle("Attention gamma trajectories (no learnable softmax temperature exists in these configurations)")
_save(fig, "09b_gamma_trajectories.png")

# ---- extra insight plots ----
fig, ax = plt.subplots(figsize=(8, 5))
for n in order:
    d = epoch_df[epoch_df.config == n]
    ax.plot(d.epoch, d.train_acc - d.val_acc, marker="o", ms=4, color=COLORS[n], label=n)
ax.set_title("Generalisation gap: train acc (30-way) − val acc (5-way)\n(different task difficulty, so read the TREND, not the absolute value)")
ax.set_xlabel("epoch"); ax.set_ylabel("percentage points"); ax.set_xticks(range(1, MAX_EPOCHS + 1)); ax.legend(ncol=2, fontsize=9)
_save(fig, "10_generalisation_gap.png")

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([RESULTS[n]["per_episode_acc"] for n in order], showmeans=True, patch_artist=True)
for patch, n in zip(bp["boxes"], order):
    patch.set_facecolor(COLORS[n]); patch.set_alpha(0.6)
ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(order)
ax.set_title("Per-episode test accuracy (same 600 episodes for every config)"); ax.set_ylabel("accuracy (%)"); ax.grid(axis="x", visible=False)
_save(fig, "11_per_episode_test_accuracy_boxplot.png")

nb = [n for n in order if n != "baseline"]
if nb:
    sub = main_df[main_df.config.isin(nb)]
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.errorbar(sub.delta_vs_baseline, range(len(nb)), xerr=sub.delta_ci95_paired, fmt="o", capsize=5, color="k")
    for i, (dv, c) in enumerate(zip(sub.delta_vs_baseline, sub.config)):
        ax.plot(dv, i, "o", ms=9, color=COLORS[c])
    ax.axvline(0, color="grey", ls="--"); ax.set_yticks(range(len(nb))); ax.set_yticklabels(nb); ax.invert_yaxis()
    ax.set_xlabel("Δ test accuracy vs baseline (percentage points, paired over identical episodes, 95% CI)")
    ax.set_title("Is the change distinguishable from the baseline?\n(CI covers episode sampling only — not training-seed variance)")
    _save(fig, "12_paired_difference_vs_baseline.png")

groups = ["stem", "layer1", "layer2", "layer3", "layer4"]
fig, ax = plt.subplots(figsize=(9, 5)); x = np.arange(len(groups)); w = 0.8 / len(order)
for j, n in enumerate(order):
    ax.bar(x + (j - (len(order) - 1) / 2) * w, [RESULTS[n]["stats"][f"backbone_rel_drift_{g}"] for g in groups], w, label=n, color=COLORS[n])
ax.set_xticks(x); ax.set_xticklabels(groups); ax.set_ylabel("‖θ − θ_pretrained‖ / ‖θ_pretrained‖")
ax.set_title("How far each backbone block drifted from the ImageNet weights (tested checkpoint)"); ax.legend(ncol=2, fontsize=8); ax.grid(axis="x", visible=False)
_save(fig, "13_backbone_drift_from_pretrained.png")

In [ ]:
# ============================================================================
# 10. ZIP (no automatic download - fetch it manually from Kaggle's Output / Files panel)
# ============================================================================
bad = [p for p in RESULTS_DIR.rglob("*") if p.suffix in (".pth", ".pt", ".ckpt", ".bin")]
assert not bad, f"unexpected checkpoint files: {bad}"
zip_path = Path(shutil.make_archive(str(WORK_DIR / "msenet_ablation_results"), "zip", root_dir=str(RESULTS_DIR))).resolve()
print("Files in the ZIP:")
for p in sorted(RESULTS_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(RESULTS_DIR)}  ({p.stat().st_size / 1024:.1f} KB)")
print(f"\nZIP created ({zip_path.stat().st_size / 1e6:.2f} MB) at:\n  {zip_path}")